# Chapter 11 — Training: Which Link in the Learning Chain Is Broken?

**Book alignment:** PyTorch From First Principles, Chapter 11

**Question this notebook isolates:** When forward, loss, backward, and `optimizer.step()` all run, which learning-chain link (task alignment, objective, dependency, ownership, update, capability) first fails to produce its consequence?

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — Healthy reference: a learnable task must actually learn

Control problem: linearly separable rule, full-batch Adam. A healthy run starts near `ln(2)` and reaches near-zero loss with perfect accuracy.

In [ ]:
RULE_W = torch.tensor([1.0, 0.75])

def known_rule(x):
    return ((x @ RULE_W) > 0).long()

def make_data(n=512, seed=0):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 2, generator=g)
    return x, known_rule(x)

class TinyClassifier(nn.Module):
    def __init__(self, hidden=16):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU())
        self.head = nn.Linear(hidden, 2)
    def forward(self, x):
        return self.head(self.features(x))

x, y = make_data()
maj = float(torch.bincount(y).max() / len(y))
torch.manual_seed(42)
model = TinyClassifier()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(200):
    opt.zero_grad()
    loss = F.cross_entropy(model(x), y)
    loss.backward()
    opt.step()
with torch.no_grad():
    final_loss = float(F.cross_entropy(model(x), y))
    acc = float((model(x).argmax(-1) == y).float().mean())
print(f'majority={maj:.3f} final loss={final_loss:.4f} acc={acc:.4f}')

In [ ]:
assert abs(float(F.cross_entropy(torch.zeros(4, 2), torch.zeros(4, dtype=torch.long))) - 0.6931) < 1e-3
assert final_loss < 0.05
assert acc > 0.97
print('healthy learning chain verified')

## 2 — Stale optimizer: gradients exist but owned parameters never move

Build the optimizer, then replace the head. The new head gets gradients but the optimizer owns only stale objects, so its delta is exactly zero and its grad buffer accumulates.

In [ ]:
def opt_ids(opt):
    return {id(p) for g in opt['groups'] for p in g['params']} if isinstance(opt, dict) else {id(p) for g in opt.param_groups for p in g['params']}

torch.manual_seed(42)
m2 = TinyClassifier()
opt2 = torch.optim.Adam(m2.parameters(), lr=1e-2)
m2.head = nn.Linear(16, 2)
owned = opt_ids(opt2)
missing = [n for n, p in m2.named_parameters() if p.requires_grad and id(p) not in owned]
print('trainable but unowned:', missing)
before = {n: p.detach().clone() for n, p in m2.named_parameters()}
for it in range(3):
    opt2.zero_grad()
    F.cross_entropy(m2(x), y).backward()
    gnorm = float(m2.head.weight.grad.norm())
    opt2.step()
    print(f'iter {it + 1} head grad norm={gnorm:.4f}')
deltas = {n: float((p.detach() - before[n]).norm()) for n, p in m2.named_parameters()}
print('head delta:', deltas['head.weight'], 'old-feat delta:', deltas['features.0.weight'])

In [ ]:
assert set(missing) == {'head.weight', 'head.bias'}
assert deltas['head.weight'] == 0.0 and deltas['head.bias'] == 0.0
assert m2.head.weight.grad is not None
print('stale-optimizer link verified')

## 3 — Objective floor and capability: softmax-before-CE stalls, overfit proves capacity

Feeding probabilities into `F.cross_entropy` floors the loss near 0.313 even for perfect outputs, while a fresh model overfits one tiny batch to near zero — the capability link.

In [ ]:
perfect = torch.tensor([[1.0, 0.0]])
floor = float(F.cross_entropy(perfect, torch.tensor([0])))
print('perfect-probs CE floor:', round(floor, 4))

class Detached(TinyClassifier):
    def forward(self, x):
        return self.head(self.features(x).detach())

torch.manual_seed(42)
md = Detached()
md.zero_grad()
F.cross_entropy(md(x[:32]), y[:32]).backward()
print('detached feat grad is None:', md.features[0].weight.grad is None)
print('detached head grad is None:', md.head.weight.grad is None)

torch.manual_seed(7)
m3 = TinyClassifier()
o3 = torch.optim.Adam(m3.parameters(), lr=1e-2)
xb, yb = x[:32], y[:32]
for step in range(200):
    o3.zero_grad()
    F.cross_entropy(m3(xb), yb).backward()
    o3.step()
with torch.no_grad():
    tiny = float(F.cross_entropy(m3(xb), yb))
print('tiny-batch loss:', round(tiny, 4))

In [ ]:
assert abs(floor - 0.3133) < 1e-3
assert md.features[0].weight.grad is None
assert md.head.weight.grad is not None
assert tiny < 0.05
print('objective floor + capability verified')

## What we earned

`loss.backward()` proves a derivative was computed, not that the intended object will move; `optimizer.step()` proves a method ran, not that it owned the right objects. Walk fixed-batch consequences in order — task, objective, dependency, gradient, ownership, update, capability — and fix the first missing one.

Chapter 12 keeps the learned model fixed and asks the other question: at what cost did it learn — which phase owns the step time and memory?